# 교재 코드 기반 시선 추정 파이프라인

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paragon0730/gaze_estimation/blob/main/pipeline.ipynb)

『밑바닥부터 시작하는 딥러닝』 교재 코드(`common/`)만으로 MPIIGaze 시선 추정 CNN을 만들고 leave-one-person-out으로 평가함. PyTorch 등 프레임워크는 쓰지 않음.

| 단계 | 교재 | 이 저장소 |
|------|------|-----------|
| 계층 (Affine, ReLU, Convolution, Pooling, Dropout) | 5장, 7장 `common/layers.py` | 그대로 사용 |
| im2col | 7장 `common/util.py` | 그대로 사용 (dtype만 유지) |
| 회귀 손실, head pose 결합 | 5장 계산 그래프 | `gaze/layers.py` (`MeanSquaredLoss`, `Concat`) 새로 작성 |
| 모델 | 7장 `SimpleConvNet` | `gaze/model.py` `GazeLeNet` |
| 기울기 확인 | 5장 `gradient_check` | 아래 2절 |
| 옵티마이저, 초깃값 | 6장 Adam, He 초깃값 | `common/optimizer.py` |
| 학습 루프 | 4장 `train_neuralnet`, `common/trainer.py` | `gaze/trainer.py` |
| 필터·출력 시각화 | 7장 `visualize_filter_output` | 아래 6절 |

목차
1. 데이터와 fold 분할
2. 모델 구조와 기울기 확인
3. 한 fold 학습 (테스트 p00)
4. 평가
5. 오차 분석
6. 필터와 특징 맵 시각화
7. 15-fold 전체 결과

In [ ]:
def is_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

if is_colab():
    !git clone https://github.com/paragon0730/gaze_estimation.git
    %cd gaze_estimation
    !pip -q install opencv-python scipy
    !apt-get -qq install -y fonts-nanum > /dev/null
    !python -m preprocess.mpii

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common.util import smooth_curve
from gaze import viz
from gaze.data import PERSONS, Standardizer, load_all, split_fold, stack
from gaze.experiment import CONFIG, make_set, run_fold
from gaze.metrics import angular_error
from gaze.model import GazeLeNet
from gaze.trainer import evaluate

viz.setup()
pd.set_option("display.precision", 2)

## 1. 데이터와 fold 분할

- 테스트: 한 사람 (3,000개)
- 검증: 학습용 14명 중 테스트 다음 번호 한 명 → 에폭 선택(early stopping)에만 씀
- 학습: 나머지 13명 (39,000개)
- 표준화 통계(이미지 평균·표준편차, head·gaze의 평균·표준편차)는 **학습 13명으로만** 계산함

In [ ]:
data = load_all()
TEST = "p00"
train_p, val_p, test_p = split_fold(TEST)
print("train:", train_p, "\nval:", val_p, "\ntest:", test_p)

tr = stack(data, train_p)
std = Standardizer().fit(*tr)
train_set = make_set(std, *tr)
val_set = make_set(std, *stack(data, val_p))
test_set = make_set(std, *data[test_p])
print({k: v.shape for k, v in train_set.items()})
print(f"image mean/std {std.eye_mean:.3f}/{std.eye_std:.3f}, gaze mean {np.degrees(std.gaze_mean).round(2)}°, "
      f"gaze std {np.degrees(std.gaze_std).round(2)}°")

## 2. 모델 구조와 기울기 확인

### 2.1 GazeLeNet

MPIIGaze 논문(Zhang et al., CVPR 2015)의 LeNet 구조를 교재 계층으로 그대로 쌓음. head pose는 마지막 완전연결 계층 직전에 이어 붙임.

```
eye (1,36,60) → Conv(20,5×5) → ReLU → Pool(2) → Conv(50,5×5) → ReLU → Pool(2)
             → Affine(500) → ReLU ─┐
head (2) ──────────────────────────┴→ Concat(502) → Affine(2) → MeanSquaredLoss
```

In [ ]:
net = GazeLeNet(conv_params=CONFIG["conv_params"], hidden_size=CONFIG["hidden_size"], seed=0)
x = train_set["x"][:4]
print(f"{'input':10s} {x.shape}")
for name, layer in net.layers.items():
    x = layer.forward(x)
    print(f"{name:10s} {x.shape}")
x = net.concat.forward(x, train_set["head"][:4]); print(f"{'Concat':10s} {x.shape}")
x = net.out_layer.forward(x); print(f"{'Affine2':10s} {x.shape}")

n_params = {k: v.size for k, v in net.params.items()}
print(n_params, "\n총 매개변수:", sum(n_params.values()))

### 2.2 기울기 확인 (5장 gradient check)

오차역전파로 구한 기울기와 수치 미분 기울기를 비교함. 수치 미분은 느리므로 같은 구조의 작은 망(입력 12×16, 필터 3·4개)에서 float64로 확인함. 새로 만든 `Concat`, `MeanSquaredLoss`의 backward도 함께 검증됨.

In [ ]:
small = GazeLeNet(input_dim=(1, 12, 16), conv_params=((3, 3), (4, 3)), hidden_size=6, seed=1, dtype=np.float64)
rng = np.random.default_rng(0)
xs, hs, ts = rng.standard_normal((4, 1, 12, 16)), rng.standard_normal((4, 2)), rng.standard_normal((4, 2))
_, grad_bp = small.gradient(xs, hs, ts)
grad_num = small.numerical_gradient(xs, hs, ts)
for k in grad_bp:
    print(f"{k}: {grad_bp[k].shape}  max |backprop - numerical| = {np.abs(grad_bp[k] - grad_num[k]).max():.2e}")

## 3. 한 fold 학습 (테스트 p00)

하이퍼파라미터는 모든 fold에 같게 둠 (`gaze/experiment.py`의 `CONFIG`). 에폭마다 검증 사람의 각도 오차를 재고, 가장 낮은 에폭의 매개변수를 최종 모델로 씀.

GitHub Actions에서 이미 학습한 결과(`results/params/p00.pkl`, `results/lopo/p00.json`)가 있으면 불러오고, 없으면 여기서 학습함 (CPU에서 수십 분).

In [ ]:
print(json.dumps(CONFIG))
param_path, hist_path = f"results/params/{TEST}.pkl", f"results/lopo/{TEST}.json"
if os.path.exists(param_path) and os.path.exists(hist_path):
    net = GazeLeNet(conv_params=CONFIG["conv_params"], hidden_size=CONFIG["hidden_size"], seed=0)
    net.load_params(param_path)
    result = json.load(open(hist_path))
    print("학습된 매개변수를 불러옴:", param_path)
else:
    result, net, _, _ = run_fold(data, TEST, return_model=True)
hist = result["history"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
it = np.array(hist["iter_loss"])
axes[0].plot(it, color="#c7d7ee", lw=0.6, label="iteration")
axes[0].plot(smooth_curve(it), color=viz.BLUE, lw=1.5, label="smooth_curve (교재 util)")
axes[0].set(ylim=(0, np.percentile(it, 99)), xlabel="iteration", ylabel="MSE loss (표준화된 gaze)", title="학습 손실")
axes[0].legend(frameon=False)

ep = np.arange(1, len(hist["train_err"]) + 1)
axes[1].plot(ep, hist["train_err"], "o-", color=viz.BLUE, label="train (3,000개 표본)")
axes[1].plot(ep, hist["val_err"], "s-", color=viz.ORANGE, label=f"val {result['val']}")
axes[1].axvline(hist["best_epoch"], color=viz.GRAY, ls="--", lw=1)
axes[1].annotate(f"선택된 에폭 {hist['best_epoch']}", (hist["best_epoch"], max(hist["val_err"])),
                 xytext=(4, 0), textcoords="offset points", fontsize=8, color="#555")
axes[1].set(xlabel="epoch", ylabel="각도 오차 (°)", title="에폭별 각도 오차")
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. 평가

테스트 사람 p00의 3,000개에 대한 각도 오차. README의 기준선(평균 gaze 예측, ridge 회귀)과 비교함.

In [ ]:
test_err, pred = evaluate(net, std, test_set["x"], test_set["head"], test_set["gaze"])
gt = test_set["gaze"]
err = angular_error(pred, gt)
mean_err = angular_error(np.broadcast_to(tr[2].mean(0), gt.shape), gt).mean()
print(f"[{TEST}] GazeLeNet 평균 {test_err:.2f}°, 중앙값 {np.median(err):.2f}°")
print(f"[{TEST}] 평균 gaze 예측 {mean_err:.2f}°, ridge 회귀(README) 5.31°")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].hist(err, bins=60, color=viz.BLUE)
axes[0].axvline(err.mean(), color=viz.ORANGE, lw=1.5)
axes[0].annotate(f"평균 {err.mean():.2f}°", (err.mean(), axes[0].get_ylim()[1] * 0.9), xytext=(4, 0),
                 textcoords="offset points", fontsize=8)
axes[0].set(title="샘플별 각도 오차", xlabel="도", ylabel="샘플 수")
for ax, k, name in zip(axes[1:], [0, 1], ["yaw", "pitch"]):
    g, p = np.degrees(gt[:, k]), np.degrees(pred[:, k])
    ax.scatter(g, p, s=3, alpha=0.3, color=viz.BLUE, edgecolors="none")
    lim = [min(g.min(), p.min()), max(g.max(), p.max())]
    ax.plot(lim, lim, color=viz.GRAY, lw=1, ls="--")
    r = np.corrcoef(g, p)[0, 1]
    ax.set(title=f"gaze {name}: 정답 vs 예측 (r={r:.2f})", xlabel="정답 (°)", ylabel="예측 (°)")
    ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## 5. 오차 분석

### 5.1 어디서 틀리는가

gaze가 평균에서 멀수록(극단적인 방향일수록) 오차가 커지는지, 왼눈·오른눈 차이가 있는지 봄.

In [ ]:
meta = pd.read_csv(f"data/EyeBased/MPIIGaze/Label/{TEST}.label", sep=" ")
dev = angular_error(np.broadcast_to(tr[2].mean(0), gt.shape), gt)  # 평균 gaze로부터 떨어진 정도
bins = np.quantile(dev, np.linspace(0, 1, 7))
which = np.clip(np.digitize(dev, bins[1:-1]), 0, 5)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].bar(range(6), [err[which == b].mean() for b in range(6)], color=viz.BLUE, width=0.7)
axes[0].set_xticks(range(6), [f"{bins[b]:.0f}–{bins[b+1]:.0f}°" for b in range(6)])
axes[0].set(title="평균 gaze로부터의 거리별 오차", xlabel="정답 gaze와 평균 gaze 사이 각도", ylabel="평균 오차 (°)")
eyes = meta["WhichEye"].values
axes[1].bar(["left", "right"], [err[eyes == "left"].mean(), err[eyes == "right"].mean()],
            color=[viz.BLUE, viz.ORANGE], width=0.5)
axes[1].set(title="눈별 평균 오차", ylabel="평균 오차 (°)")
plt.tight_layout(); plt.show()

### 5.2 잘 맞힌 예와 틀린 예

초록 = 정답, 주황 = 예측.

In [ ]:
order = np.argsort(err)
fig, axes = plt.subplots(2, 6, figsize=(13, 3))
for row, idx in zip(axes, [order[:6], order[-6:]]):
    for ax, i in zip(row, idx):
        img = data[TEST][0][i] / 255.0
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        viz.draw_gaze(ax, img, gt[i], "lime")
        viz.draw_gaze(ax, img, pred[i], viz.ORANGE)
        ax.set_title(f"{err[i]:.1f}°", fontsize=8); ax.axis("off")
axes[0, 0].text(-8, 18, "best", rotation=90, va="center", fontsize=9)
axes[1, 0].text(-8, 18, "worst", rotation=90, va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 6. 필터와 특징 맵 시각화 (7장)

### 6.1 첫 번째 합성곱 계층의 필터

학습 전(He 초깃값, 무작위)과 학습 후 비교. 학습 후에는 경계·명암 변화를 잡는 구조가 생김.

In [ ]:
def show_filters(W, title):
    n = W.shape[0]
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(n * 0.45, 1.3))
    for ax, i in zip(axes.ravel(), range(n)):
        ax.imshow(W[i, 0], cmap="gray", interpolation="nearest"); ax.axis("off")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=9); plt.show()

init = GazeLeNet(conv_params=CONFIG["conv_params"], hidden_size=CONFIG["hidden_size"], seed=CONFIG["seed"])
show_filters(init.params["W1"], "학습 전 conv1 필터 (5×5 × 20)")
show_filters(net.params["W1"], "학습 후 conv1 필터")

### 6.2 특징 맵 (visualize_filter_output)

테스트 이미지 하나를 넣었을 때 conv1(ReLU 후)과 conv2(ReLU 후) 출력.

In [ ]:
i = order[len(order) // 2]
x = test_set["x"][i:i + 1]
outs = {}
for name, layer in net.layers.items():
    x = layer.forward(x)
    if name in ("Relu1", "Relu2"):
        outs[name] = x[0]

fig = plt.figure(figsize=(12, 1.6))
ax = fig.add_subplot(1, 1, 1); ax.imshow(data[TEST][0][i], cmap="gray"); ax.axis("off"); ax.set_title("입력")
plt.show()
for name, n_show, ncol in [("Relu1", 20, 10), ("Relu2", 20, 10)]:
    fmap = outs[name]
    fig, axes = plt.subplots(n_show // ncol, ncol, figsize=(12, 1.0 * n_show // ncol))
    for ax, k in zip(axes.ravel(), range(n_show)):
        ax.imshow(fmap[k], cmap="gray"); ax.axis("off")
    fig.suptitle(f"{name} 출력 {fmap.shape} 중 {n_show}개", fontsize=9); plt.show()

## 7. 15-fold 전체 결과

`python train_lopo.py`로 15개 fold를 모두 학습한 결과 (`results/lopo_results.csv`). GitHub Actions에서 fold마다 병렬로 실행함 (`.github/workflows/pipeline.yml`).

In [ ]:
csv = "results/lopo_results.csv"
if os.path.exists(csv):
    res = pd.read_csv(csv).set_index("person")
    display(pd.concat([res, res.mean(numeric_only=True).to_frame("평균").T]).T)
    fig, ax = plt.subplots(figsize=(12, 3.5))
    xs = np.arange(len(res))
    w = 0.27
    for k, (col, name, color) in enumerate([("mean_gaze", "평균 gaze 예측", viz.GRAY),
                                            ("ridge", "ridge 회귀", viz.ORANGE),
                                            ("cnn", "GazeLeNet", viz.BLUE)]):
        ax.bar(xs + (k - 1) * w, res[col], width=w - 0.03, color=color,
               label=f"{name} (평균 {res[col].mean():.2f}°)")
    ax.set_xticks(xs, res.index)
    ax.set(ylabel="각도 오차 (°)", title="leave-one-person-out 각도 오차")
    ax.legend(frameon=False, ncol=3, loc="upper left")
    plt.tight_layout(); plt.show()
else:
    print("results/lopo_results.csv가 없음. python train_lopo.py 후 python train_lopo.py --summarize 실행")